In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # E5.1 · Data Quality — Bronze
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Validar técnicamente los datos ingeridos en Bronze antes
# MAGIC de iniciar las transformaciones de Silver.
# MAGIC
# MAGIC ### Principio
# MAGIC
# MAGIC **Bronze conserva; DQ observa; Silver corrige.**
# MAGIC
# MAGIC Este notebook:
# MAGIC
# MAGIC - NO elimina registros.
# MAGIC - NO deduplica.
# MAGIC - NO corrige valores.
# MAGIC - NO modifica los datos Bronze.
# MAGIC
# MAGIC Únicamente:
# MAGIC
# MAGIC - ejecuta reglas;
# MAGIC - calcula métricas;
# MAGIC - registra resultados;
# MAGIC - permite detener el pipeline ante fallas críticas.


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import Row
from pyspark.sql import functions as F

import uuid

from datetime import (
    datetime,
    timezone
)


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "yelp_bronze"

GOV_SCHEMA = "yelp_gov"

LAYER = "bronze"


DQ_RULE_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "dq_reglas"
)


DQ_RESULT_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "dq_resultados"
)


dq_execution_id = str(
    uuid.uuid4()
)


dq_execution_ts = (
    datetime
    .now(timezone.utc)
)


print("=" * 70)
print("DATA QUALITY - BRONZE")
print("=" * 70)

print(
    f"Execution ID : "
    f"{dq_execution_id}"
)

print(
    f"Layer        : "
    f"{LAYER}"
)

print("=" * 70)


# COMMAND ----------

# ============================================================
# 3. VALIDAR SCHEMA DE GOBIERNO
# ============================================================

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{GOV_SCHEMA}
    """
)


print(
    f"✔ Schema disponible: "
    f"{CATALOG}.{GOV_SCHEMA}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 4. Catálogo de reglas Bronze
# MAGIC
# MAGIC Las reglas Bronze se concentran en:
# MAGIC
# MAGIC - Completitud.
# MAGIC - Unicidad.
# MAGIC - Validez técnica.
# MAGIC - Trazabilidad.
# MAGIC
# MAGIC Las reglas de negocio y consistencia semántica más complejas
# MAGIC serán evaluadas posteriormente en Silver.


# COMMAND ----------

# ============================================================
# 4. DEFINICIÓN DE REGLAS
# ============================================================

reglas_bronze = [

    # ========================================================
    # BUSINESS
    # ========================================================

    Row(
        regla_id="DQ-BR-001",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_business",
        columna="*",
        dimension="completitud",
        tipo_regla="TABLE_NOT_EMPTY",
        descripcion="Business debe contener registros",
        expresion_sql=None,
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-002",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_business",
        columna="business_id",
        dimension="completitud",
        tipo_regla="CONDITION",
        descripcion="business_id debe estar informado",
        expresion_sql="business_id IS NOT NULL",
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-003",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_business",
        columna="business_id",
        dimension="unicidad",
        tipo_regla="UNIQUENESS",
        descripcion="business_id debe ser único",
        expresion_sql=None,
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-004",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_business",
        columna="_metadata",
        dimension="trazabilidad",
        tipo_regla="CONDITION",
        descripcion="Business debe poseer metadata técnica de linaje",
        expresion_sql="""
            _archivo_origen IS NOT NULL
            AND _execution_id IS NOT NULL
            AND _record_checksum IS NOT NULL
            AND _schema_hash IS NOT NULL
        """,
        criticidad="alta",
        umbral_pct=100.0
    ),


    # ========================================================
    # USER
    # ========================================================

    Row(
        regla_id="DQ-BR-005",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_user",
        columna="*",
        dimension="completitud",
        tipo_regla="TABLE_NOT_EMPTY",
        descripcion="User debe contener registros",
        expresion_sql=None,
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-006",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_user",
        columna="user_id",
        dimension="completitud",
        tipo_regla="CONDITION",
        descripcion="user_id debe estar informado",
        expresion_sql="user_id IS NOT NULL",
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-007",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_user",
        columna="user_id",
        dimension="unicidad",
        tipo_regla="UNIQUENESS",
        descripcion="user_id debe ser único",
        expresion_sql=None,
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-008",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_user",
        columna="_metadata",
        dimension="trazabilidad",
        tipo_regla="CONDITION",
        descripcion="User debe poseer metadata técnica de linaje",
        expresion_sql="""
            _archivo_origen IS NOT NULL
            AND _execution_id IS NOT NULL
            AND _record_checksum IS NOT NULL
            AND _schema_hash IS NOT NULL
        """,
        criticidad="alta",
        umbral_pct=100.0
    ),


    # ========================================================
    # REVIEW
    # ========================================================

    Row(
        regla_id="DQ-BR-009",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_review",
        columna="*",
        dimension="completitud",
        tipo_regla="TABLE_NOT_EMPTY",
        descripcion="Review debe contener registros",
        expresion_sql=None,
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-010",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_review",
        columna="review_id",
        dimension="completitud",
        tipo_regla="CONDITION",
        descripcion="review_id debe estar informado",
        expresion_sql="review_id IS NOT NULL",
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-011",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_review",
        columna="review_id",
        dimension="unicidad",
        tipo_regla="UNIQUENESS",
        descripcion="review_id debe ser único",
        expresion_sql=None,
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-012",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_review",
        columna="business_id",
        dimension="completitud",
        tipo_regla="CONDITION",
        descripcion="Toda Review debe contener business_id",
        expresion_sql="business_id IS NOT NULL",
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-013",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_review",
        columna="user_id",
        dimension="completitud",
        tipo_regla="CONDITION",
        descripcion="Toda Review debe contener user_id",
        expresion_sql="user_id IS NOT NULL",
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-014",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_review",
        columna="stars",
        dimension="validez",
        tipo_regla="CONDITION",
        descripcion="El rating original debe encontrarse entre 1 y 5",
        expresion_sql="try_cast(stars AS DOUBLE) BETWEEN 1 AND 5",
        criticidad="alta",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-015",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_review",
        columna="_metadata",
        dimension="trazabilidad",
        tipo_regla="CONDITION",
        descripcion="Review debe poseer metadata técnica de linaje",
        expresion_sql="""
            _archivo_origen IS NOT NULL
            AND _execution_id IS NOT NULL
            AND _record_checksum IS NOT NULL
            AND _schema_hash IS NOT NULL
        """,
        criticidad="alta",
        umbral_pct=100.0
    ),


    # ========================================================
    # TIP
    # ========================================================

    Row(
        regla_id="DQ-BR-016",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_tip",
        columna="*",
        dimension="completitud",
        tipo_regla="TABLE_NOT_EMPTY",
        descripcion="Tip debe contener registros",
        expresion_sql=None,
        criticidad="media",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-017",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_tip",
        columna="business_id",
        dimension="completitud",
        tipo_regla="CONDITION",
        descripcion="Tip debe contener business_id",
        expresion_sql="business_id IS NOT NULL",
        criticidad="media",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-018",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_tip",
        columna="_metadata",
        dimension="trazabilidad",
        tipo_regla="CONDITION",
        descripcion="Tip debe poseer metadata técnica de linaje",
        expresion_sql="""
            _archivo_origen IS NOT NULL
            AND _execution_id IS NOT NULL
            AND _record_checksum IS NOT NULL
            AND _schema_hash IS NOT NULL
        """,
        criticidad="alta",
        umbral_pct=100.0
    ),


    # ========================================================
    # CHECKIN
    # ========================================================

    Row(
        regla_id="DQ-BR-019",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_checkin",
        columna="*",
        dimension="completitud",
        tipo_regla="TABLE_NOT_EMPTY",
        descripcion="Check-in debe contener registros",
        expresion_sql=None,
        criticidad="media",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-020",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_checkin",
        columna="business_id",
        dimension="completitud",
        tipo_regla="CONDITION",
        descripcion="Check-in debe contener business_id",
        expresion_sql="business_id IS NOT NULL",
        criticidad="media",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-021",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_checkin",
        columna="date",
        dimension="completitud",
        tipo_regla="CONDITION",
        descripcion="Check-in debe contener fechas",
        expresion_sql="date IS NOT NULL",
        criticidad="media",
        umbral_pct=100.0
    ),

    Row(
        regla_id="DQ-BR-022",
        capa="bronze",
        tabla_objetivo="workspace.yelp_bronze.brz_checkin",
        columna="_metadata",
        dimension="trazabilidad",
        tipo_regla="CONDITION",
        descripcion="Check-in debe poseer metadata técnica de linaje",
        expresion_sql="""
            _archivo_origen IS NOT NULL
            AND _execution_id IS NOT NULL
            AND _record_checksum IS NOT NULL
            AND _schema_hash IS NOT NULL
        """,
        criticidad="alta",
        umbral_pct=100.0
    )
]


df_reglas_bronze = (
    spark.createDataFrame(
        reglas_bronze
    )
)


display(
    df_reglas_bronze
)


# COMMAND ----------

# ============================================================
# 5. CREAR CATÁLOGO CENTRAL DE REGLAS
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {DQ_RULE_TABLE}
    (
        regla_id STRING,
        capa STRING,
        tabla_objetivo STRING,
        columna STRING,
        dimension STRING,
        tipo_regla STRING,
        descripcion STRING,
        expresion_sql STRING,
        criticidad STRING,
        umbral_pct DOUBLE
    )
    USING DELTA
    """
)


# COMMAND ----------

# ============================================================
# 6. UPSERT DE REGLAS BRONZE
# ============================================================

df_reglas_bronze.createOrReplaceTempView(
    "src_dq_reglas_bronze"
)


spark.sql(
    f"""
    MERGE INTO {DQ_RULE_TABLE} AS target

    USING src_dq_reglas_bronze AS source

    ON target.regla_id = source.regla_id

    WHEN MATCHED THEN
      UPDATE SET *

    WHEN NOT MATCHED THEN
      INSERT *
    """
)


print(
    "✔ Catálogo DQ Bronze actualizado."
)


# COMMAND ----------

# ============================================================
# 7. CREAR TABLA HISTÓRICA DE RESULTADOS
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {DQ_RESULT_TABLE}
    (
        ejecucion_id STRING,
        ejecucion_ts TIMESTAMP,
        regla_id STRING,
        capa STRING,
        tabla_objetivo STRING,

        columna STRING,

        dimension STRING,

        tipo_regla STRING,

        criticidad STRING,
        filas_totales BIGINT,
        filas_ok BIGINT,
        filas_error BIGINT,
        pct_cumplimiento DOUBLE,
        umbral_pct DOUBLE,
        paso BOOLEAN
    )
    USING DELTA
    """
)

try:
    spark.sql(
        f"""
        ALTER TABLE {DQ_RESULT_TABLE}
        ADD COLUMNS
        (
            ejecucion_id STRING,
            capa STRING,
            dimension STRING,
            criticidad STRING,
            filas_error BIGINT,
            umbral_pct DOUBLE
        )
        """
    )
except Exception:
    pass


# COMMAND ----------

# MAGIC %md
# MAGIC # 8. Motor genérico de Data Quality


# COMMAND ----------

# ============================================================
# 8. MOTOR GENÉRICO DE EVALUACIÓN
# ============================================================

def evaluar_reglas_bronze():

    resultados = []

    reglas = (
        spark
        .table(
            DQ_RULE_TABLE
        )
        .filter(
            F.col("capa")
            == "bronze"
        )
        .collect()
    )


    for r in reglas:

        print()
        print(
            f"Ejecutando {r.regla_id}: "
            f"{r.descripcion}"
        )


        # ----------------------------------------------------
        # Tabla
        # ----------------------------------------------------

        df = spark.table(
            r.tabla_objetivo
        )


        total = df.count()


        # ----------------------------------------------------
        # TABLE NOT EMPTY
        # ----------------------------------------------------

        if (
            r.tipo_regla
            == "TABLE_NOT_EMPTY"
        ):

            if total > 0:

                filas_ok = total

                pct = 100.0

            else:

                filas_ok = 0

                pct = 0.0


        # ----------------------------------------------------
        # UNIQUENESS
        # ----------------------------------------------------

        elif (
            r.tipo_regla
            == "UNIQUENESS"
        ):

            filas_ok = (
                df

                .select(
                    r.columna
                )

                .distinct()

                .count()
            )


            pct = (
                round(
                    100.0
                    * filas_ok
                    / total,
                    2
                )

                if total > 0

                else 0.0
            )


        # ----------------------------------------------------
        # CONDITION
        # ----------------------------------------------------

        elif (
            r.tipo_regla
            == "CONDITION"
        ):

            filas_ok = (
                spark.sql(
                    f"""
                    SELECT
                        COUNT(*) AS c

                    FROM
                        {r.tabla_objetivo}

                    WHERE
                        {r.expresion_sql}
                    """
                )
                .first()
                .c
            )


            pct = (
                round(
                    100.0
                    * filas_ok
                    / total,
                    2
                )

                if total > 0

                else 0.0
            )


        else:

            raise Exception(
                f"Tipo de regla no soportado: "
                f"{r.tipo_regla}"
            )


        # ----------------------------------------------------
        # Resultado
        # ----------------------------------------------------

        filas_error = (
            total
            - filas_ok
        )


        paso = (
            pct
            >= r.umbral_pct
        )


        resultados.append(
            (
                dq_execution_id,
                dq_execution_ts,
                r.regla_id,
                r.capa,
                r.tabla_objetivo,
                r.dimension,
                r.criticidad,
                total,
                filas_ok,
                filas_error,
                pct,
                r.umbral_pct,
                paso
            )
        )


        icon = (
            "✔"
            if paso
            else "❌"
        )


        print(
            f"{icon} "
            f"{pct}% "
            f"(umbral {r.umbral_pct}%)"
        )


    # --------------------------------------------------------
    # DataFrame resultados
    # --------------------------------------------------------

    df_resultados = (
        spark.createDataFrame(
            resultados,
            """
            ejecucion_id STRING,
            ejecucion_ts TIMESTAMP,
            regla_id STRING,
            capa STRING,
            tabla_objetivo STRING,
            dimension STRING,
            criticidad STRING,
            filas_totales BIGINT,
            filas_ok BIGINT,
            filas_error BIGINT,
            pct_cumplimiento DOUBLE,
            umbral_pct DOUBLE,
            paso BOOLEAN
            """
        )
    )


    # --------------------------------------------------------
    # Persistencia histórica
    # --------------------------------------------------------

    (
        df_resultados

        .write

        .format(
            "delta"
        )

        .mode(
            "append"
        )

        .saveAsTable(
            DQ_RESULT_TABLE
        )
    )


    return df_resultados


# COMMAND ----------

# ============================================================
# 9. EJECUTAR DQ
# ============================================================

resultado_dq = (
    evaluar_reglas_bronze()
)


display(
    resultado_dq
    .orderBy(
        "regla_id"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 10. Scorecard Bronze


# COMMAND ----------

# ============================================================
# 10. SCORECARD
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            dimension,

            COUNT(*) AS reglas,

            SUM(
                CASE
                    WHEN paso
                    THEN 1
                    ELSE 0
                END
            ) AS reglas_ok,

            ROUND(
                AVG(
                    pct_cumplimiento
                ),
                2
            ) AS cumplimiento_promedio

        FROM
            {DQ_RESULT_TABLE}

        WHERE
            ejecucion_id = '{dq_execution_id}'

        GROUP BY
            dimension

        ORDER BY
            dimension
        """
    )
)


# COMMAND ----------

# ============================================================
# 11. DETALLE DE FALLAS
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            regla_id,

            tabla_objetivo,

            dimension,

            criticidad,

            filas_totales,

            filas_error,

            pct_cumplimiento,

            umbral_pct,

            paso

        FROM
            {DQ_RESULT_TABLE}

        WHERE
            ejecucion_id = '{dq_execution_id}'

        ORDER BY
            paso ASC,
            criticidad,
            regla_id
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 12. Quality Gate Bronze
# MAGIC
# MAGIC Si una regla de criticidad **alta** no alcanza su umbral,
# MAGIC el pipeline no debe continuar hacia Silver.


# COMMAND ----------

# ============================================================
# 12. QUALITY GATE
# ============================================================

fallas_criticas = (
    spark.sql(
        f"""
        SELECT
            COUNT(*) AS c

        FROM
            {DQ_RESULT_TABLE}

        WHERE
            ejecucion_id = '{dq_execution_id}'

            AND criticidad = 'alta'

            AND paso = false
        """
    )
    .first()
    .c
)


if fallas_criticas > 0:

    raise Exception(
        f"""
        ❌ QUALITY GATE BRONZE FALLÓ

        Se detectaron {fallas_criticas}
        reglas críticas incumplidas.

        El pipeline NO continuará hacia Silver.

        Consultar:
        {DQ_RESULT_TABLE}
        """
    )


print(
    "✔ Quality Gate Bronze superado."
)

print(
    "✔ El pipeline puede continuar hacia Silver."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Definition of Done — DQ Bronze
# MAGIC
# MAGIC - [ ] Cinco entidades Bronze evaluadas.
# MAGIC - [ ] Completitud evaluada.
# MAGIC - [ ] Unicidad de identificadores evaluada.
# MAGIC - [ ] Validez técnica evaluada.
# MAGIC - [ ] Metadata de trazabilidad evaluada.
# MAGIC - [ ] Resultados persistidos históricamente.
# MAGIC - [ ] Quality Gate implementado.
# MAGIC - [ ] Ningún registro Bronze fue eliminado o modificado.